In [1]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

NOAA_API  = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
NOAA_META = "https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/{}.json"
YEARS = [2025, 2026]

In [2]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

STATION_IDS = ["8518750", "8516945"]  # The Battery, Kings Point


def _noaa_session():
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def station_meta(station_id):
    r = _noaa_session().get(NOAA_META.format(station_id), timeout=30)
    r.raise_for_status()
    s = r.json()["stations"][0]
    return {"name": s["name"], "lat": float(s["lat"]), "lng": float(s["lng"])}


def fetch_predictions(station_id, year):
    """Fetch high/low predictions one month at a time to avoid API timeouts."""
    session = _noaa_session()
    months = []
    for month in range(1, 13):
        begin = pd.Timestamp(year, month, 1)
        end   = begin + pd.offsets.MonthEnd(0)
        r = session.get(NOAA_API, params={
            "product":    "predictions",
            "begin_date": begin.strftime("%Y%m%d"),
            "end_date":   end.strftime("%Y%m%d"),
            "datum":      "MLLW",
            "station":    station_id,
            "time_zone":  "lst_ldt",
            "interval":   "hilo",
            "units":      "english",
            "format":     "json",
        }, timeout=30)
        r.raise_for_status()
        months.append(pd.DataFrame(r.json()["predictions"]))
    return pd.concat(months, ignore_index=True)


frames = []
for sid in STATION_IDS:
    meta = station_meta(sid)
    df   = pd.concat([fetch_predictions(sid, yr) for yr in YEARS], ignore_index=True)

    df["datetime"]     = pd.to_datetime(df["t"], format="%Y-%m-%d %H:%M")
    df["pred_in_ft"]   = df["v"].astype(float)
    df["pred_in_cm"]   = (df["pred_in_ft"] * 30.48).round().astype(int)
    df["date"]         = df["datetime"].dt.strftime("%Y/%m/%d")
    df["day"]          = df["datetime"].dt.strftime("%a")
    df["time"]         = df["datetime"].dt.strftime("%I:%M %p")
    df["highlow"]      = df["type"]
    df["station_id"]   = sid
    df["station_name"] = meta["name"]
    df["geometry"]     = Point(meta["lng"], meta["lat"])

    frames.append(df[["datetime", "date", "day", "time",
                       "pred_in_ft", "pred_in_cm", "highlow",
                       "station_id", "station_name", "geometry"]])

tides_gdf = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326",
)

In [3]:
for sid, group in tides_gdf.groupby("station_id", sort=False):
    print(f"\n── {group['station_name'].iloc[0]}  ({sid})  "
          f"{group['geometry'].iloc[0].y:.4f}°N, {group['geometry'].iloc[0].x:.4f}°W ──")
    display(group[["date", "day", "time", "pred_in_ft", "pred_in_cm", "highlow"]]
              .head(8)
              .reset_index(drop=True))

print(f"\nTotal records: {len(tides_gdf)}  |  CRS: {tides_gdf.crs}")

tides_gdf.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\tide_predictions.geojson",
    driver="GeoJSON",
)
print("Saved → tide_predictions.geojson")


── The Battery  (8518750)  40.7006°N, -74.0142°W ──


,date,day,time,pred_in_ft,pred_in_cm,highlow
0,2025/01/01,Wed,02:45 AM,-0.088,-3,L
1,2025/01/01,Wed,08:39 AM,5.172,158,H
2,2025/01/01,Wed,03:33 PM,-0.528,-16,L
3,2025/01/01,Wed,09:12 PM,4.010,122,H
4,2025/01/02,Thu,03:28 AM,-0.147,-4,L
5,2025/01/02,Thu,09:23 AM,5.121,156,H
6,2025/01/02,Thu,04:13 PM,-0.559,-17,L
7,2025/01/02,Thu,09:59 PM,4.094,125,H



── Kings Point  (8516945)  40.8103°N, -73.7649°W ──


,date,day,time,pred_in_ft,pred_in_cm,highlow
0,2025/01/01,Wed,05:45 AM,0.035,1,L
1,2025/01/01,Wed,11:39 AM,8.003,244,H
2,2025/01/01,Wed,06:30 PM,-0.677,-21,L
3,2025/01/02,Thu,12:15 AM,7.060,215,H
4,2025/01/02,Thu,06:28 AM,-0.105,-3,L
5,2025/01/02,Thu,12:23 PM,8.057,246,H
6,2025/01/02,Thu,07:07 PM,-0.741,-23,L
7,2025/01/03,Fri,12:58 AM,7.254,221,H



Total records: 5641  |  CRS: EPSG:4326
Saved → tide_predictions.geojson


## USGS Tidal Observations (parameter 62620 — water surface elevation above NAVD 1988)

In [4]:
import dataretrieval.nwis as nwis

USGS_SITES = {
    "01311875": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01311850": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01311145": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01376562": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet
}

START_DT = "2025-01-01"
END_DT = pd.Timestamp.today().strftime("%Y-%m-%d")


def fetch_usgs_meta(site_no):
    df = nwis.get_record(sites=site_no, service="site")
    row = df.iloc[0]
    return {
        "name": row["station_nm"],
        "lat":  float(row["dec_lat_va"]),
        "lon":  float(row["dec_long_va"]),
    }


def fetch_usgs_wl(site_no, param, start_dt, end_dt):
    """Fetch USGS instantaneous values one month at a time to avoid gateway timeouts."""
    start, end = pd.Timestamp(start_dt), pd.Timestamp(end_dt)
    chunks = []
    cursor = start
    while cursor <= end:
        chunk_end = min(cursor + pd.DateOffset(months=1) - pd.Timedelta(days=1), end)
        df, _ = nwis.get_iv(
            sites=site_no, parameterCd=param,
            startDT=cursor.strftime("%Y-%m-%d"),
            endDT=chunk_end.strftime("%Y-%m-%d"),
        )
        if not df.empty:
            value_col = next(
                c for c in df.columns
                if (c == param or c.startswith(param + "_")) and not c.endswith("_cd")
            )
            df = df[[value_col]].rename(columns={value_col: "value"})
            df["value"] = pd.to_numeric(df["value"], errors="coerce")
            df.index.name = "datetime"
            chunks.append(df.reset_index())
        cursor += pd.DateOffset(months=1)
    return pd.concat(chunks, ignore_index=True)


frames = []
for site_no, param_cd in USGS_SITES.items():
    meta = fetch_usgs_meta(site_no)
    df   = fetch_usgs_wl(site_no, param_cd, START_DT, END_DT)
    df["site_no"]      = site_no
    df["param_cd"]     = param_cd
    df["station_name"] = meta["name"]
    df["geometry"]     = Point(meta["lon"], meta["lat"])
    frames.append(df)

usgs_gdf = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326",
)

C:\Users\sriva\AppData\Local\Temp\ipykernel_3008\800185757.py:15: DeprecationWarning: `nwis.get_record` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use the appropriate `waterdata.get_*()` for the service you need instead.
  df = nwis.get_record(sites=site_no, service="site")
C:\Users\sriva\AppData\Local\Temp\ipykernel_3008\800185757.py:31: DeprecationWarning: `nwis.get_iv` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_continuous()` instead.
  df, _ = nwis.get_iv(
C:\Users\sriva\AppData\Local\Temp\ipykernel_3008\800185757.py:31: DeprecationWarning: `nwis.get_iv` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_continuous()` instead.
  df, _ = nwis.get_iv(
C:\Users\sriva\AppData\Local\Temp\ipykernel_3008\800185757.py:31: DeprecationWarning: `nwis.get_iv` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_con

In [5]:
for site, group in usgs_gdf.groupby("site_no", sort=False):
    print()
    print(f"── {group['station_name'].iloc[0]}  ({site})  param {group['param_cd'].iloc[0]}  "
          f"{group['geometry'].iloc[0].y:.4f}°N, {group['geometry'].iloc[0].x:.4f}°W ──")
    display(group[["datetime", "value"]].head(8).reset_index(drop=True))

print()
print(f"Total records: {len(usgs_gdf)}  |  CRS: {usgs_gdf.crs}")

usgs_gdf.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\usgs_water_levels.geojson",
    driver="GeoJSON",
)
print("Saved → usgs_water_levels.geojson")


── ROCKAWAY INLET NEAR FLOYD BENNETT FIELD NY  (01311875)  param 62620  40.5742°N, -73.8855°W ──


,datetime,value
0,2025-01-01 05:00:00+00:00,0.46
1,2025-01-01 05:06:00+00:00,0.36
2,2025-01-01 05:12:00+00:00,0.25
3,2025-01-01 05:18:00+00:00,0.13
4,2025-01-01 05:24:00+00:00,0.02
5,2025-01-01 05:30:00+00:00,-0.07
6,2025-01-01 05:36:00+00:00,-0.21
7,2025-01-01 05:42:00+00:00,-0.31



── JAMAICA BAY AT INWOOD NY  (01311850)  param 62620  40.6175°N, -73.7576°W ──


,datetime,value
0,2025-01-01 05:00:00+00:00,0.34
1,2025-01-01 05:06:00+00:00,0.30
2,2025-01-01 05:12:00+00:00,0.22
3,2025-01-01 05:18:00+00:00,0.12
4,2025-01-01 05:24:00+00:00,0.05
5,2025-01-01 05:30:00+00:00,0.01
6,2025-01-01 05:36:00+00:00,-0.07
7,2025-01-01 05:42:00+00:00,-0.20



── EAST ROCKAWAY INLET AT ATLANTIC BEACH NY  (01311145)  param 62620  40.5930°N, -73.7374°W ──


,datetime,value
0,2025-01-01 05:00:00+00:00,0.24
1,2025-01-01 05:06:00+00:00,0.06
2,2025-01-01 05:12:00+00:00,-0.10
3,2025-01-01 05:18:00+00:00,-0.22
4,2025-01-01 05:24:00+00:00,-0.29
5,2025-01-01 05:30:00+00:00,-0.46
6,2025-01-01 05:36:00+00:00,-0.41
7,2025-01-01 05:42:00+00:00,-0.62



── GREAT KILLS HARBOR AT GREAT KILLS NY  (01376562)  param 62620  40.5389°N, -74.1299°W ──


,datetime,value
0,2025-01-01 05:00:00+00:00,0.57
1,2025-01-01 05:06:00+00:00,0.56
2,2025-01-01 05:12:00+00:00,0.44
3,2025-01-01 05:18:00+00:00,0.31
4,2025-01-01 05:24:00+00:00,0.20
5,2025-01-01 05:30:00+00:00,0.13
6,2025-01-01 05:36:00+00:00,0.04
7,2025-01-01 05:42:00+00:00,-0.11



Total records: 521023  |  CRS: EPSG:4326
Saved → usgs_water_levels.geojson


## Standardized Unified Schema

Both sources are aligned to a common schema:

| column | type | notes |
|---|---|---|
| `datetime` | datetime (UTC, tz-aware) | NOAA localized from US/Eastern; USGS converted to UTC |
| `water_level_ft` | float | **NAVD88 datum** for both sources |
| `tidal_phase` | string | `H` / `L` (NOAA); `rising` / `falling` (USGS) |
| `surge_ft` | float | `0.0` for NOAA predicted; `NaN` for USGS (requires paired predicted) |
| `source` | string | `"noaa_predicted"` or `"usgs_observed"` |
| `station_id` | string | NOAA station ID or USGS site number |
| `station_name` | string | — |
| `geometry` | Point (EPSG:4326) | — |

**Datum conversion (NOAA only):** NOAA predictions are in feet above MLLW. NOAA's own datums API gives the height of each datum above the station datum (STND); the difference `MLLW_stnd − NAVD_stnd` equals how far MLLW is above NAVD88. Subtracting that offset converts every NOAA reading to feet above NAVD88.

In [6]:
def fetch_noaa_datums(station_id):
    r = requests.get(
        f"https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/{station_id}/datums.json",
        params={"units": "english"},
    )
    r.raise_for_status()
    return {d["name"]: float(d["value"]) for d in r.json()["datums"]}


# How far MLLW sits above NAVD88 at each station (feet).
# Negative at NYC stations — MLLW is below NAVD88.
# Adding this to a MLLW-referenced value gives a NAVD88-referenced value.
MLLW_ABOVE_NAVD = {}
for sid in STATION_IDS:
    d = fetch_noaa_datums(sid)
    MLLW_ABOVE_NAVD[sid] = d["MLLW"] - d["NAVD88"]
    print(f"{sid}: MLLW is {MLLW_ABOVE_NAVD[sid]:+.3f} ft above NAVD88")

8518750: MLLW is -2.770 ft above NAVD88
8516945: MLLW is -4.210 ft above NAVD88


### Datum conversion: MLLW → NAVD88

**What the NOAA datums API returns**

`fetch_noaa_datums()` returns every datum for a station as a height **above STND** — the physical benchmark bolt at the gauge. STND is just a local reference pin; it has no global meaning. Both `d["MLLW"]` and `d["NAVD88"]` are measured up from the same STND, so their difference is a meaningful physical offset between the two datums.

**Physical ordering at NYC stations**

NAVD88 approximates mean sea level. MLLW (Mean Lower Low Water) is the average of the lowest daily tides — well below mean sea level. So at every NYC gauge:

```
↑  MHW    (~+2 ft relative to NAVD88)
↑  NAVD88  ≈ mean sea level          d["NAVD88"] > d["MLLW"]
↑  MLLW   (~-2.5 ft relative to NAVD88)
```

Therefore `MLLW_ABOVE_NAVD = d["MLLW"] − d["NAVD88"]` is a **negative number** — it quantifies how far below NAVD88 the MLLW zero mark sits.

**Converting a MLLW-referenced prediction to NAVD88**

NOAA tide predictions (`pred_in_ft`) are heights above the MLLW zero mark. To convert, find the water surface's absolute elevation above STND, then re-reference it to NAVD88:

```
absolute elevation above STND  =  d["MLLW"] + pred_in_ft
elevation above NAVD88         = (d["MLLW"] + pred_in_ft) − d["NAVD88"]
                               =  pred_in_ft + (d["MLLW"] − d["NAVD88"])
                               =  pred_in_ft + MLLW_ABOVE_NAVD
```

Because `MLLW_ABOVE_NAVD` is negative, this **subtracts** the gap between NAVD88 and MLLW — which is exactly what you want. A low-tide reading of 0.3 ft above MLLW becomes `0.3 + (−2.5) = −2.2 ft` relative to NAVD88, correctly placing low water below mean sea level.

**Why `− MLLW_ABOVE_NAVD` is wrong**

Subtracting a negative number adds its magnitude: `pred − (−2.5) = pred + 2.5`. That pushes every reading *further above* MLLW rather than re-referencing it downward to NAVD88 — a physically impossible result for typical tide levels.

In [7]:
import numpy as np

# ── NOAA: MLLW → NAVD88, datetimes localized US/Eastern → UTC ──
noaa_rows = []
for sid, group in tides_gdf.groupby("station_id", sort=False):
    g = group.copy()
    g["water_level_ft"] = g["pred_in_ft"] + MLLW_ABOVE_NAVD[sid]
    g["datetime"] = (
        g["datetime"]
        .dt.tz_localize("US/Eastern", ambiguous="NaT", nonexistent="shift_forward")
        .dt.tz_convert("UTC")
    )
    g = g.dropna(subset=["datetime"])
    g["tidal_phase"] = g["highlow"].map({"H": "H", "L": "L"})
    g["surge_ft"]    = 0.0
    g["source"]      = "noaa_predicted"
    g["station_id"]  = sid
    noaa_rows.append(g[["datetime", "water_level_ft", "tidal_phase",
                         "surge_ft", "source", "station_id", "station_name", "geometry"]])

noaa_std = gpd.GeoDataFrame(
    pd.concat(noaa_rows, ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)

# ── USGS: all sites use 62620 (NAVD88); normalize datetimes to UTC ──
usgs_rows = []
for site, group in usgs_gdf.groupby("site_no", sort=False):
    g = group.copy().sort_values("datetime").reset_index(drop=True)
    g["water_level_ft"] = g["value"]
    dt = pd.to_datetime(g["datetime"])
    g["datetime"] = dt.dt.tz_convert("UTC") if dt.dt.tz is not None else dt.dt.tz_localize("UTC")
    dv = g["water_level_ft"].diff()
    phase = pd.array([pd.NA] * len(g), dtype="string")
    phase[dv > 0] = "rising"
    phase[dv < 0] = "falling"
    g["tidal_phase"] = phase
    # Relabel peak transitions as H/L so downstream code can filter tidal_phase == 'H'
    next_phase = g["tidal_phase"].shift(-1)
    g.loc[(g["tidal_phase"] == "rising")  & (next_phase == "falling"), "tidal_phase"] = "H"
    g.loc[(g["tidal_phase"] == "falling") & (next_phase == "rising"),  "tidal_phase"] = "L"
    g["surge_ft"]    = np.nan
    g["source"]      = "usgs_observed"
    g["station_id"]  = site
    usgs_rows.append(g[["datetime", "water_level_ft", "tidal_phase",
                         "surge_ft", "source", "station_id", "station_name", "geometry"]])

usgs_std = gpd.GeoDataFrame(
    pd.concat(usgs_rows, ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)

# ── Combine ──
tidal_unified = gpd.GeoDataFrame(
    pd.concat([noaa_std, usgs_std], ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)

tidal_unified.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\tidal_unified.geojson",
    driver="GeoJSON",
)

print(f"Schema: {list(tidal_unified.columns)}")
print(f"Sources: {tidal_unified['source'].value_counts().to_dict()}")
print(f"Total rows: {len(tidal_unified)}")
print(f"Datetime range: {tidal_unified['datetime'].min()} → {tidal_unified['datetime'].max()}")
display(tidal_unified.groupby("source").head(3).reset_index(drop=True))

Schema: ['datetime', 'water_level_ft', 'tidal_phase', 'surge_ft', 'source', 'station_id', 'station_name', 'geometry']
Sources: {'usgs_observed': 521023, 'noaa_predicted': 5639}
Total rows: 526662
Datetime range: 2025-01-01 05:00:00+00:00 → 2027-01-01 01:47:00+00:00


,datetime,water_level_ft,tidal_phase,surge_ft,source,station_id,station_name,geometry
0,2025-01-01 07:45:00+00:00,-2.858,L,0.0,noaa_predicted,8518750,The Battery,POINT (-74.01417 40.70055)
1,2025-01-01 13:39:00+00:00,2.402,H,0.0,noaa_predicted,8518750,The Battery,POINT (-74.01417 40.70055)
2,2025-01-01 20:33:00+00:00,-3.298,L,0.0,noaa_predicted,8518750,The Battery,POINT (-74.01417 40.70055)
3,2025-01-01 05:00:00+00:00,0.460,<NA>,NaN,usgs_observed,01311875,ROCKAWAY INLET NEAR FLOYD BENNETT FIELD NY,POINT (-73.8855 40.57419)
4,2025-01-01 05:06:00+00:00,0.360,falling,NaN,usgs_observed,01311875,ROCKAWAY INLET NEAR FLOYD BENNETT FIELD NY,POINT (-73.8855 40.57419)
5,2025-01-01 05:12:00+00:00,0.250,falling,NaN,usgs_observed,01311875,ROCKAWAY INLET NEAR FLOYD BENNETT FIELD NY,POINT (-73.8855 40.57419)


In [8]:
tidal_unified[tidal_unified["source"] == "usgs_observed"]["tidal_phase"].value_counts()


tidal_phase
falling    244463
rising     229451
H           15561
L           15531
Name: count, dtype: Int64

In [9]:
tidal_unified[tidal_unified['source'] == 'usgs_observed']['datetime'].agg(['min', 'max'])


min   2025-01-01 05:00:00+00:00
max   2026-06-30 19:48:00+00:00
Name: datetime, dtype: datetime64[us, UTC]